In [1]:
import os
os.environ["PYSPARK_PYTHON"] = r"C:\Users\krish\anaconda3\python.exe"
os.environ["PYSPARK_DRIVER_PYTHON"] = r"C:\Users\krish\anaconda3\python.exe"

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.ml.feature import StringIndexer, VectorAssembler, StandardScaler
from pyspark.ml.regression import LinearRegression, RandomForestRegressor
from pyspark.ml.evaluation import RegressionEvaluator

In [3]:
spark = SparkSession.builder \
                    .appName("Bengaluru House Data - Regression") \
                    .master("local") \
                    .getOrCreate()

In [4]:
spark

In [5]:
blr_data = spark.read \
                .format("csv") \
                .option("inferSchema", "true") \
                .option("header", "true") \
                .load("../Data/Bengaluru House Data.csv")

In [6]:
blr_data.show(5)

+--------------------+-------------+--------------------+---------+-------+----------+----+-------+-----+
|           area_type| availability|            location|     size|society|total_sqft|bath|balcony|price|
+--------------------+-------------+--------------------+---------+-------+----------+----+-------+-----+
|Super built-up  Area|       19-Dec|Electronic City P...|    2 BHK|Coomee |      1056|   2|      1|39.07|
|          Plot  Area|Ready To Move|    Chikka Tirupathi|4 Bedroom|Theanmp|      2600|   5|      3|120.0|
|      Built-up  Area|Ready To Move|         Uttarahalli|    3 BHK|   NULL|      1440|   2|      3| 62.0|
|Super built-up  Area|Ready To Move|  Lingadheeranahalli|    3 BHK|Soiewre|      1521|   3|      1| 95.0|
|Super built-up  Area|Ready To Move|            Kothanur|    2 BHK|   NULL|      1200|   2|      1| 51.0|
+--------------------+-------------+--------------------+---------+-------+----------+----+-------+-----+
only showing top 5 rows



In [7]:
blr_view = blr_data.createOrReplaceTempView('blr_view')

### Count the total number of housing-properties listed from 'HSR Layout' location.

In [8]:
blr_data[blr_data['location'] == 'HSR Layout'].count()

53

In [9]:
blr_data.where("location = 'HSR Layout'").count()

53

In [10]:
spark.sql('''
select count(*) as `HSR Property Count`
from blr_view
where location = "HSR Layout"
''').show()

+------------------+
|HSR Property Count|
+------------------+
|                53|
+------------------+



### How many ‘2 BHK’ size housing-properties are listed from 'Whitefield' location?

In [11]:
blr_data.where("location = 'Whitefield' and size = '2 BHK'").count()

223

In [12]:
spark.sql('''
select count(*) as `2bhk_whitefield`
from blr_view
where location = 'Whitefield'
and size = '2 BHK'
''').show()

+---------------+
|2bhk_whitefield|
+---------------+
|            223|
+---------------+



### What is the average price of ‘2 BHK’ size housing-properties in ‘HSR Layout’ location?

In [13]:
blr_data[(blr_data['location'] == 'HSR Layout') & (blr_data['size'] == '2 BHK')] \
    .select('price') \
    .agg({'price':'avg'}) \
    .show()

+----------+
|avg(price)|
+----------+
|  55.71875|
+----------+



In [14]:
blr_data.where("location = 'HSR Layout' and size = '2 BHK'") \
        .groupBy(['size', 'location']) \
        .agg(
            F.avg('price')
        ) \
        .show()

+-----+----------+----------+
| size|  location|avg(price)|
+-----+----------+----------+
|2 BHK|HSR Layout|  55.71875|
+-----+----------+----------+



In [15]:
spark.sql('''
select size, location, avg(price)
from blr_view
where location = 'HSR Layout'
and size = '2 BHK'
group by location, size
''').show()

+-----+----------+----------+
| size|  location|avg(price)|
+-----+----------+----------+
|2 BHK|HSR Layout|  55.71875|
+-----+----------+----------+



### EDA, Model Building, Training & Evaluation

### Remove the features, having more than one third of their entries as missing/null. 
### For the remaining missing values-remove the corresponding row entry from the DataFrame.

In [16]:
blr_data.select([F.round(F.count(F.when(F.col(feature).isNull(), feature))*100/blr_data.count(), 3).alias(feature) for feature in blr_data.columns]).show()

+---------+------------+--------+----+-------+----------+-----+-------+-----+
|area_type|availability|location|size|society|total_sqft| bath|balcony|price|
+---------+------------+--------+----+-------+----------+-----+-------+-----+
|      0.0|         0.0|   0.008|0.12| 41.306|       0.0|0.548|  4.572|  0.0|
+---------+------------+--------+----+-------+----------+-----+-------+-----+



In [17]:
blr_data = blr_data.drop("society")

In [18]:
blr_data = blr_data.withColumn("total_sqft_int", F.col("total_sqft").cast("float"))

In [19]:
blr_data = blr_data.dropna()

In [20]:
blr_data.select([F.round(F.count(F.when(F.col(feature).isNull(), feature))*100/blr_data.count(), 3).alias(feature) for feature in blr_data.columns]).show()

+---------+------------+--------+----+----------+----+-------+-----+--------------+
|area_type|availability|location|size|total_sqft|bath|balcony|price|total_sqft_int|
+---------+------------+--------+----+----------+----+-------+-----+--------------+
|      0.0|         0.0|     0.0| 0.0|       0.0| 0.0|    0.0|  0.0|           0.0|
+---------+------------+--------+----+----------+----+-------+-----+--------------+



In [21]:
blr_data.printSchema()

root
 |-- area_type: string (nullable = true)
 |-- availability: string (nullable = true)
 |-- location: string (nullable = true)
 |-- size: string (nullable = true)
 |-- total_sqft: string (nullable = true)
 |-- bath: integer (nullable = true)
 |-- balcony: integer (nullable = true)
 |-- price: double (nullable = true)
 |-- total_sqft_int: float (nullable = true)



### Convert all string columns into numeric values using StringIndexer transformer and make sure now Data Frame does not have any string columns anymore.

In [22]:
string_indexer = StringIndexer(inputCols=["area_type", "availability", "location", "size"], 
                               outputCols=["area_type_ind", "availability_ind", "location_ind", "size_ind"])

In [23]:
blr_data_ind = string_indexer.fit(blr_data).transform(blr_data)
blr_data_ind = blr_data_ind.drop("area_type", "availability", "location", "size")

In [24]:
blr_data_ind.show(2)

+----------+----+-------+-----+--------------+-------------+----------------+------------+--------+
|total_sqft|bath|balcony|price|total_sqft_int|area_type_ind|availability_ind|location_ind|size_ind|
+----------+----+-------+-----+--------------+-------------+----------------+------------+--------+
|      1056|   2|      1|39.07|        1056.0|          0.0|             5.0|        14.0|     0.0|
|      2600|   5|      3|120.0|        2600.0|          2.0|             0.0|       179.0|     2.0|
+----------+----+-------+-----+--------------+-------------+----------------+------------+--------+
only showing top 2 rows



###  Using vector Assembler combine all columns (except target column i.e., 'price') of spark Data Frame into single column (name as features). 
### Make sure Data Frame now contains only two columns features and price.

In [25]:
vector_assembler = VectorAssembler(inputCols=["total_sqft_int", "bath", "balcony", "total_sqft_int", "area_type_ind", "availability_ind", "location_ind", "size_ind"], 
                                   outputCol="features")

In [26]:
blr_data_assemb = vector_assembler.transform(blr_data_ind)
blr_data_assemb = blr_data_assemb.drop("total_sqft", "bath", "balcony", "total_sqft_int", "area_type_ind", "availability_ind", "location_ind", "size_ind")

In [27]:
blr_data_assemb.show(2)

+-----+--------------------+
|price|            features|
+-----+--------------------+
|39.07|[1056.0,2.0,1.0,1...|
|120.0|[2600.0,5.0,3.0,2...|
+-----+--------------------+
only showing top 2 rows



In [28]:
blr_data_assemb.printSchema()

root
 |-- price: double (nullable = true)
 |-- features: vector (nullable = true)



In [29]:
train_data, test_data = blr_data_assemb.randomSplit([0.75, 0.25], seed=0)

In [30]:
print("Train data size:", train_data.count())
print("Test data size:", test_data.count())

Train data size: 9334
Test data size: 3190


In [31]:
lr = LinearRegression(featuresCol="features", labelCol="price")

In [32]:
lr_model = lr.fit(train_data)

In [33]:
test_pred = lr_model.transform(test_data)

In [34]:
evaluator = RegressionEvaluator(labelCol="price", predictionCol="prediction")

In [35]:
evaluator.evaluate(test_pred, {evaluator.metricName: "rmse"})

107.98244654956612

### Random Forest Regressor

In [36]:
rf = RandomForestRegressor(featuresCol="features", labelCol="price", maxBins=1500)

In [37]:
evaluator = RegressionEvaluator(labelCol="price", predictionCol="prediction", metricName="rmse")

In [38]:
rf_model = rf.fit(train_data)

In [39]:
train_pred_rf = rf_model.transform(train_data)

In [40]:
evaluator.evaluate(train_pred_rf)

74.14799283838906

In [41]:
test_pred_rf = rf_model.transform(test_data)

In [42]:
evaluator.evaluate(test_pred_rf, {evaluator.metricName: "rmse"})

146.70195273602576